# B04b — Expert Systems and Knowledge Bases

**COMPSCI 713 — Week 4: MYCIN and Rule-Based AI**

---

## Overview

Expert systems were the dominant AI paradigm of the 1970s–1980s. MYCIN (Stanford, 1972) could diagnose bacterial infections as well as human specialists — using only IF-THEN rules and certainty factors. This lesson builds a working expert system from scratch.

In this lesson you will:
- Understand the architecture of an expert system
- Build a knowledge base with IF-THEN rules
- Implement forward chaining and backward chaining inference
- Handle uncertainty with certainty factors (MYCIN-style)
- Build a mini medical diagnosis expert system
- Understand the limits that led to the AI winter

### COMPSCI 713 Alignment
- **Week 4 Monday:** MYCIN (expert systems)

## 1. Expert System Architecture

An expert system has three core components:

```
┌─────────────────────────────────────────────┐
│              Expert System                  │
│                                             │
│  ┌──────────────┐    ┌──────────────────┐   │
│  │ Knowledge    │    │  Inference       │   │
│  │ Base         │◄──►│  Engine          │   │
│  │ (IF-THEN     │    │  (Forward /      │   │
│  │  rules)      │    │   Backward       │   │
│  └──────────────┘    │   Chaining)      │   │
│                      └────────┬─────────┘   │
│  ┌──────────────┐             │             │
│  │ Working      │◄────────────┘             │
│  │ Memory       │                           │
│  │ (known facts)│                           │
│  └──────────────┘                           │
└─────────────────────────────────────────────┘
         ▲                    │
         │ facts              │ conclusions
         │                    ▼
      [User]           [Explanation]
```

- **Knowledge Base:** domain expertise encoded as IF-THEN rules
- **Working Memory:** current facts known about the problem
- **Inference Engine:** applies rules to derive new facts
- **Explanation facility:** tells the user *why* a conclusion was reached

In [ ]:
# ── Simple Forward-Chaining Expert System ──────────────────────────────────

class Rule:
    """An IF-THEN rule: IF all conditions are true THEN assert conclusion."""
    def __init__(self, conditions, conclusion, explanation=''):
        self.conditions  = conditions   # list of strings
        self.conclusion  = conclusion   # string
        self.explanation = explanation

    def __repr__(self):
        conds = ' AND '.join(self.conditions)
        return f"IF {conds} THEN {self.conclusion}"


class ForwardChainingES:
    """Forward-chaining (data-driven) expert system."""

    def __init__(self, rules):
        self.rules = rules

    def run(self, facts: set, verbose=True):
        working_memory = set(facts)
        fired = []

        changed = True
        while changed:
            changed = False
            for rule in self.rules:
                if (all(c in working_memory for c in rule.conditions)
                        and rule.conclusion not in working_memory):
                    working_memory.add(rule.conclusion)
                    fired.append(rule)
                    changed = True
                    if verbose:
                        print(f"  FIRED: {rule}")
                        if rule.explanation:
                            print(f"         → {rule.explanation}")

        return working_memory, fired


# ── Animal Kingdom Knowledge Base ──────────────────────────────────────────
animal_rules = [
    Rule(['has_hair'],                          'is_mammal',    'Hair is a mammal characteristic'),
    Rule(['gives_milk'],                        'is_mammal',    'Milk production is mammalian'),
    Rule(['has_feathers'],                      'is_bird',      'Feathers define birds'),
    Rule(['flies', 'lays_eggs'],                'is_bird',      'Flying egg-layers are birds'),
    Rule(['is_mammal', 'eats_meat'],            'is_carnivore', 'Meat-eating mammal'),
    Rule(['is_mammal', 'has_pointed_teeth'],    'is_carnivore', 'Pointed teeth indicate carnivore'),
    Rule(['is_mammal', 'has_hooves'],           'is_ungulate',  'Hooves indicate ungulate'),
    Rule(['is_carnivore', 'has_tawny_color', 'has_dark_spots'], 'is_cheetah', 'Spotted tawny carnivore'),
    Rule(['is_carnivore', 'has_tawny_color', 'has_black_stripes'], 'is_tiger', 'Striped tawny carnivore'),
    Rule(['is_ungulate', 'has_long_neck'],      'is_giraffe',   'Long-necked ungulate'),
    Rule(['is_ungulate', 'has_black_stripes'],  'is_zebra',     'Striped ungulate'),
    Rule(['is_bird', 'cannot_fly'],             'is_penguin',   'Non-flying bird'),
]

es = ForwardChainingES(animal_rules)

# Test: what animal has hair, eats meat, tawny color, dark spots?
print("=== Animal Identification ===")
print("Known facts: has_hair, eats_meat, has_tawny_color, has_dark_spots")
print()
facts = {'has_hair', 'eats_meat', 'has_tawny_color', 'has_dark_spots'}
result, fired_rules = es.run(facts)
print(f"\nConclusions: {result - facts}")

## 2. Backward Chaining — Goal-Directed Reasoning

Forward chaining: start with facts, derive conclusions (data-driven).

Backward chaining: start with a **goal**, work backwards to find supporting facts (goal-driven). This is how MYCIN worked — it started with a hypothesis ("does this patient have organism X?") and gathered evidence to confirm or deny it.

In [ ]:
class BackwardChainingES:
    """Backward-chaining (goal-driven) expert system."""

    def __init__(self, rules):
        self.rules = rules
        self.known_facts = set()
        self.depth = 0

    def prove(self, goal, verbose=True):
        indent = '  ' * self.depth
        if verbose:
            print(f"{indent}? Trying to prove: {goal}")

        if goal in self.known_facts:
            if verbose:
                print(f"{indent}✓ Already known: {goal}")
            return True

        for rule in self.rules:
            if rule.conclusion == goal:
                if verbose:
                    print(f"{indent}  Trying rule: {rule}")
                self.depth += 1
                if all(self.prove(cond, verbose) for cond in rule.conditions):
                    self.known_facts.add(goal)
                    self.depth -= 1
                    if verbose:
                        print(f"{indent}✓ PROVED: {goal}")
                    return True
                self.depth -= 1

        return False


# Simpler rules for backward chaining demo
bc_rules = [
    Rule(['has_hair'],               'is_mammal'),
    Rule(['is_mammal', 'eats_meat'], 'is_carnivore'),
    Rule(['is_carnivore', 'has_dark_spots', 'has_tawny_color'], 'is_cheetah'),
]

bc_es = BackwardChainingES(bc_rules)
bc_es.known_facts = {'has_hair', 'eats_meat', 'has_dark_spots', 'has_tawny_color'}

print("=== Backward Chaining: Is this a cheetah? ===")
result = bc_es.prove('is_cheetah')
print(f"\nResult: {'YES — it is a cheetah' if result else 'Cannot prove it is a cheetah'}")

## 3. MYCIN-Style Certainty Factors

Real-world knowledge is uncertain. MYCIN introduced **Certainty Factors (CF)** — a simple way to handle uncertainty without full probability theory.

- CF ranges from **-1** (definitely false) to **+1** (definitely true)
- CF = 0 means no evidence either way
- CF > 0.2 is typically considered actionable

**Combining CFs:**
- Sequential rules: `CF_combined = CF_rule × CF_evidence`
- Parallel evidence (both support same conclusion):
  - `CF = CF1 + CF2 × (1 - CF1)` if both positive
  - `CF = CF1 + CF2 × (1 + CF1)` if both negative

In [ ]:
class CFRule:
    """Rule with a certainty factor."""
    def __init__(self, conditions, conclusion, cf):
        self.conditions = conditions
        self.conclusion = conclusion
        self.cf = cf  # certainty factor of this rule

    def __repr__(self):
        conds = ' AND '.join(f"{c[0]}(CF={c[1]:.2f})" for c in self.conditions)
        return f"IF {conds} THEN {self.conclusion} (CF={self.cf:.2f})"


def combine_cf(cf1, cf2):
    """Combine two certainty factors supporting the same conclusion."""
    if cf1 >= 0 and cf2 >= 0:
        return cf1 + cf2 * (1 - cf1)
    elif cf1 < 0 and cf2 < 0:
        return cf1 + cf2 * (1 + cf1)
    else:
        return (cf1 + cf2) / (1 - min(abs(cf1), abs(cf2)))


# Mini medical diagnosis (MYCIN-inspired)
# Evidence: (symptom, certainty_of_symptom)
patient_evidence = {
    'fever':           0.9,
    'stiff_neck':      0.8,
    'headache':        0.7,
    'photophobia':     0.6,
    'positive_culture': 0.5,
}

# Rules: IF symptom(s) THEN diagnosis (with rule CF)
medical_rules = [
    CFRule([('fever', 0.9), ('stiff_neck', 0.8)],    'bacterial_meningitis', 0.7),
    CFRule([('fever', 0.9), ('headache', 0.7)],       'viral_meningitis',     0.5),
    CFRule([('stiff_neck', 0.8), ('photophobia', 0.6)], 'bacterial_meningitis', 0.6),
    CFRule([('positive_culture', 0.5)],               'bacterial_meningitis', 0.9),
]

# Calculate CF for each diagnosis
diagnosis_cfs = {}
for rule in medical_rules:
    # CF of evidence = min of individual evidence CFs
    cf_evidence = min(patient_evidence.get(cond, 0) for cond, _ in rule.conditions)
    # CF of conclusion = rule CF × evidence CF
    cf_conclusion = rule.cf * cf_evidence

    if rule.conclusion in diagnosis_cfs:
        diagnosis_cfs[rule.conclusion] = combine_cf(diagnosis_cfs[rule.conclusion], cf_conclusion)
    else:
        diagnosis_cfs[rule.conclusion] = cf_conclusion

print("=== MYCIN-Style Medical Diagnosis ===")
print(f"Patient symptoms: {list(patient_evidence.keys())}")
print()
for diagnosis, cf in sorted(diagnosis_cfs.items(), key=lambda x: -x[1]):
    confidence = 'HIGH' if cf > 0.6 else 'MEDIUM' if cf > 0.3 else 'LOW'
    print(f"  {diagnosis:<30} CF={cf:.3f}  [{confidence}]")

best = max(diagnosis_cfs, key=diagnosis_cfs.get)
print(f"\n→ Most likely diagnosis: {best} (CF={diagnosis_cfs[best]:.3f})")

## 4. Knowledge Representation Formats

Expert systems need structured ways to represent knowledge beyond simple rules.

In [ ]:
# ── Frames (object-oriented knowledge representation) ──────────────────────

class Frame:
    """A frame represents a concept with slots (attributes) and values."""
    def __init__(self, name, parent=None, **slots):
        self.name   = name
        self.parent = parent
        self.slots  = slots

    def get(self, slot):
        """Get slot value, inheriting from parent if not defined here."""
        if slot in self.slots:
            return self.slots[slot]
        if self.parent:
            return self.parent.get(slot)
        return None

    def __repr__(self):
        parent_name = self.parent.name if self.parent else 'None'
        return f"Frame({self.name}, parent={parent_name}, slots={self.slots})"


# Build a frame hierarchy for animals
animal = Frame('Animal',
    breathes='air',
    has_cells=True,
    locomotion='unknown'
)

mammal = Frame('Mammal', parent=animal,
    has_hair=True,
    warm_blooded=True,
    gives_birth='live_young'
)

dog = Frame('Dog', parent=mammal,
    sound='bark',
    locomotion='quadruped',
    diet='omnivore'
)

# Inheritance in action
print("=== Frame Inheritance ===")
print(f"Dog.sound:        {dog.get('sound')}")
print(f"Dog.has_hair:     {dog.get('has_hair')}      (inherited from Mammal)")
print(f"Dog.breathes:     {dog.get('breathes')}       (inherited from Animal)")
print(f"Dog.warm_blooded: {dog.get('warm_blooded')}      (inherited from Mammal)")
print(f"Dog.locomotion:   {dog.get('locomotion')}  (overrides Animal default)")

## 5. Why Expert Systems Failed (and What We Learned)

Expert systems dominated AI in the 1980s but led to the **AI Winter** of the late 1980s–1990s:

| Problem | Description |
|---|---|
| **Knowledge acquisition bottleneck** | Encoding expert knowledge is slow and expensive |
| **Brittleness** | Fail completely outside their narrow domain |
| **No learning** | Cannot improve from experience |
| **Maintenance** | Rules become inconsistent as domain evolves |
| **Uncertainty** | Certainty factors are ad-hoc, not principled |

**What we learned:**
- Machine learning (decision trees, neural networks) can *induce* rules from data
- Bayesian networks provide principled uncertainty handling
- Modern LLMs can act as flexible knowledge bases
- Hybrid systems (neuro-symbolic AI) combine the best of both worlds

## 6. Summary

### Key Takeaways
- Expert systems encode domain knowledge as **IF-THEN rules** in a knowledge base
- **Forward chaining** (data-driven) vs **backward chaining** (goal-driven) inference
- **Certainty factors** handle uncertainty without full probability theory
- **Frames** provide object-oriented knowledge representation with inheritance
- Expert systems were powerful but brittle — the knowledge acquisition bottleneck led to the AI winter
- Decision trees (B04a) solved the learning problem: rules induced from data

### Next Steps
- **B04a** — Decision Trees and XGBoost (learning rules from data)
- **B05** — Neural Networks (learning representations from data)
- **I14** — Explainable AI (making black-box models interpretable)